In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [ ]:
train = pd.read_csv('data/nba_players_train.csv')
test = pd.read_csv('data/nba_players_test.csv')

train

,name,gp,min,pts,fgm,fga,fg,3p_made,3pa,3p,...,fta,ft,oreb,dreb,reb,ast,stl,blk,tov,target_5yrs
0,James Young,31,10.7,3.4,1.2,3.3,35.3,0.5,2.1,25.8,...,0.9,55.2,0.3,1.1,1.4,0.4,0.3,0.1,0.2,0
1,Michael Anderson,65,18.5,7.4,2.9,5.8,50.1,0.0,0.1,0.0,...,2.7,57.1,1.8,2.7,4.5,1.0,1.0,0.3,1.2,0
2,Kobe Bryant,71,15.5,7.6,2.5,5.9,41.7,0.7,1.9,37.5,...,2.3,81.9,0.7,1.2,1.9,1.3,0.7,0.3,1.6,1
3,Darrin Hancock,46,9.2,3.3,1.5,2.6,56.2,0.0,0.1,33.3,...,0.8,41.0,0.3,0.8,1.2,0.7,0.4,0.1,0.7,0
4,Brent Price,68,12.6,3.9,1.5,4.1,35.8,0.1,0.7,16.7,...,1.0,79.4,0.4,1.1,1.5,2.3,0.8,0.0,1.3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
933,Terry Dehere,64,11.9,5.3,2.0,5.3,37.7,0.4,0.9,40.4,...,1.3,75.3,0.4,0.7,1.1,1.2,0.4,0.1,0.9,1
934,Jordan Hamilton,26,9.9,4.4,1.8,4.3,43.2,0.7,1.8,36.2,...,0.2,40.0,0.4,2.0,2.4,0.8,0.1,0.1,0.6,1
935,David Lee,67,16.9,5.1,2.0,3.4,59.6,0.0,0.0,0.0,...,1.8,57.7,1.6,2.9,4.5,0.6,0.5,0.3,0.8,0
936,Travis Mays,64,33.5,14.3,4.6,11.3,40.6,1.1,3.1,36.5,...,5.2,77.0,0.8,1.9,2.8,4.0,1.3,0.2,2.5,0


In [ ]:
y = train["target_5yrs"].astype(int).values

X_raw = train.drop(columns=["target_5yrs"]).copy()
X_test_raw = test.copy()

for df in (X_raw, X_test_raw):
    if "name" in df.columns:
        df.drop(columns=["name"], inplace=True)

common_cols = X_raw.columns.intersection(X_test_raw.columns)
X_raw = X_raw[common_cols].copy()
X_test_raw = X_test_raw[common_cols].copy()

full = pd.concat([X_raw, X_test_raw], axis=0, ignore_index=True)

cat_cols = full.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = [c for c in full.columns if c not in cat_cols]

if num_cols:
    full[num_cols] = SimpleImputer(strategy="median").fit_transform(full[num_cols])
if cat_cols:
    full[cat_cols] = SimpleImputer(strategy="most_frequent").fit_transform(full[cat_cols])

full = pd.get_dummies(full, columns=cat_cols, drop_first=False)

scaler = StandardScaler()
full_scaled = scaler.fit_transform(full)

n_train = len(X_raw)
X = full_scaled[:n_train]
X_test = full_scaled[n_train:]

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    solver="lbfgs",
    n_jobs=-1
)


In [ ]:
model.fit(X_tr, y_tr)
y_pred = model.predict_proba(X_val)[:, 1]

print('roc= ',roc_auc_score(y_val,y_pred))

roc=  0.7697122908390515


In [ ]:
model.fit(X, y)

test_proba = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({"target_5yrs": test_proba})

submission

,target_5yrs
0,0.666274
1,0.376298
2,0.515293
3,0.223361
4,0.720404
...,...
397,0.367674
398,0.880273
399,0.833704
400,0.498916
